# Un modelo de xG propio — y qué pasa cuando se compara con uno de producción

El xG (goles esperados) responde una pregunta simple: **dado desde dónde y cómo
se remató, ¿qué probabilidad había de que fuera gol?** Es un clasificador binario
con 221.922 filas de entrenamiento, así que como problema de machine learning es
de manual.

Lo que hace este ejercicio distinto de un tutorial es que **Understat publica su
propio xG para cada tiro**. O sea que no hay que conformarse con "mi modelo da
0.26 de log-loss": hay un modelo de producción contra el cual medirse tiro por
tiro, sobre exactamente los mismos datos.

> **Cómo terminó esto**: el modelo propio **empata** con Understat en log-loss
> (0.2597 vs 0.2589) y le **gana** en poder de discriminación (AUC 0.804 vs
> 0.799), usando solo geometría y cuatro variables categóricas. Pero el hallazgo
> que no se buscaba es otro: **el xG de Understat sobreestima los goles un 17%**
> en la última temporada, y el sesgo **crece año a año** (+7% en 2021-22 → +17%
> en 2025-26) en las cinco ligas. Se propuso una explicación —deriva del entorno,
> el modelo envejece— y **se probó que solo explica ~3 de esos 17 puntos**. El
> resto queda sin explicar, y es un buen recordatorio de que un número publicado
> por una fuente respetada no es la verdad: es otro modelo.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

from viz_theme import (SEQUENTIAL_BLUE, DIVERGING, CLUSTER_COLORS, INK,
                       apply_plotly_theme, render_plot)

apply_plotly_theme(opaque_surface=True)
pd.set_option('display.max_columns', 40)
RNG = 42

# dimensiones reglamentarias: Understat da coordenadas normalizadas 0-1
LARGO, ANCHO, PORTERIA = 105.0, 68.0, 7.32
DIVERGING_SCALE = [[0.0, DIVERGING['neg']], [0.5, DIVERGING['mid']], [1.0, DIVERGING['pos']]]

## 1. Los datos y tres decisiones de limpieza

225.442 tiros de las cinco grandes ligas, 2021-22 a 2025-26, bajados de Understat
con `fetch_shots.py`. Cada uno trae coordenadas, situación, parte del cuerpo,
resultado, y el xG que le asigna Understat.

Tres decisiones antes de modelar, y las tres cambian el resultado:

- **Los autogoles salen** (749). No son intentos del equipo que remata.
- **Los penales salen del modelo** y se tratan como constante. Son 2.771 tiros
  desde exactamente el mismo punto: si entran, el modelo aprende un pico en una
  coordenada y contamina todo su vecindario.
- **Los tiros bloqueados se quedan** (61.209). Understat los incluye y les asigna
  xG; sacarlos cambiaría la definición del objetivo y rompería la comparación.

In [2]:
d = pd.read_csv('../data/processed/shots_all_seasons.csv', encoding='utf-8-sig')
d = d[d.result != 'Own Goal'].copy()
d['gol'] = (d.result == 'Goal').astype(int)

pen = d[d.situation == 'Penalty']
print(f'PENALES ({len(pen)} tiros)')
print(f'  convierten          : {pen.gol.mean():.4f}')
print(f'  Understat les asigna: {pen.xg.mean():.4f}')

d = d[d.situation != 'Penalty'].copy()
print(f'\nTiros para el modelo: {len(d)} · tasa de gol {d.gol.mean():.4f}')

PENALES (2771 tiros)
  convierten          : 0.7882
  Understat les asigna: 0.7564

Tiros para el modelo: 221922 · tasa de gol 0.0996


### Nota sobre los penales

**Un penal vale 0.79.** Es la conversión real de los 2.771 penales de estas cinco
temporadas, y coincide con el número que se maneja habitualmente (~0.76-0.79). Al
no depender de nada —siempre el mismo punto, la misma distancia, el mismo
ángulo— no hay nada que modelar: es una constante, y así se usa.

Detalle que sale gratis al mirarlo: **Understat les asigna 0.756, por debajo de
la conversión real de 0.788**. Con 2.771 tiros, esa diferencia de 3 puntos no es
ruido — son unos 88 goles de diferencia acumulada. Es la primera señal de algo
que reaparece a lo grande en la sección 6.

## 2. Geometría: las dos features que hacen casi todo el trabajo

Cualquier modelo de xG se construye sobre dos cantidades:

- **Distancia** al centro de la portería.
- **Ángulo** que subtienden los dos palos vistos desde el punto de tiro. Es lo que
  distingue rematar de frente desde el borde del área de hacerlo desde la línea
  de fondo a la misma distancia: mismo radio, portería casi invisible.

Se calculan sobre una cancha de 105×68 m con portería de 7.32 m.

In [3]:
px, py = d.location_x * LARGO, d.location_y * ANCHO

d['dist'] = np.hypot(LARGO - px, ANCHO / 2 - py)

# ángulo por el teorema del coseno entre las rectas a cada poste
a = np.hypot(LARGO - px, ANCHO / 2 + PORTERIA / 2 - py)
b = np.hypot(LARGO - px, ANCHO / 2 - PORTERIA / 2 - py)
d['angulo'] = np.arccos(np.clip((a ** 2 + b ** 2 - PORTERIA ** 2) / (2 * a * b), -1, 1))

d['dist_y'] = (d.location_y - 0.5).abs() * ANCHO      # cuánto se remata desde el costado
d['asistido'] = d.assist_player.notna().astype(int)

# proxy de rebote/barullo: otro tiro del mismo equipo en el mismo minuto. Es
# burdo —Understat solo da resolución de minuto, y marca por igual a los dos
# tiros— pero es la única señal de secuencia que hay en estos datos.
d['rebote'] = (d.groupby(['game_id', 'team', 'minute'])['shot_id'].transform('size') > 1).astype(int)

print(f'asistidos: {d.asistido.mean():.0%} · con otro tiro en el mismo minuto: {d.rebote.mean():.0%}')
print(f'distancia: mediana {d.dist.median():.1f} m · percentil 99 {d.dist.quantile(.99):.1f} m')

asistidos: 75% · con otro tiro en el mismo minuto: 22%
distancia: mediana 16.4 m · percentil 99 35.0 m


### Por qué hace falta un modelo y no una tabla

Antes de entrenar nada, un ejemplo de por qué un `groupby` no alcanza:

In [4]:
print(d.groupby('body_part').agg(tiros=('gol', 'size'), tasa_de_gol=('gol', 'mean'),
                                 distancia_media=('dist', 'mean')).round(3).to_string())

             tiros  tasa_de_gol  distancia_media
body_part                                       
Head         39786        0.099            9.701
Left Foot    71869        0.099           19.027
Other          829        0.157            8.480
Right Foot  109438        0.100           19.560


**Los cabezazos convierten al 9.9%, exactamente igual que los pies.** Y sin
embargo cualquiera que haya visto fútbol sabe que un cabezazo es peor que un
remate con el pie desde el mismo sitio.

La tabla no miente, pero esconde: los cabezazos se rematan a **10 metros de
media**, contra 19-20 de los pies. Están más cerca *porque* son cabezazos —
llegan de centros y córners al área chica. Las dos cosas se cancelan y el
promedio crudo queda plano.

Eso es exactamente lo que un modelo desenreda: **a igual distancia y ángulo**,
¿cuánto cuesta cabecear? La respuesta está en la sección 7, y no se parece en
nada al 9.9% contra 10.0% de esta tabla.

## 3. Partición: por temporada, no al azar

Acá es donde se cae la mayoría de los tutoriales de xG. Un split aleatorio
**filtra información**: dos tiros del mismo partido —y sobre todo de la misma
jugada, que es justo lo que captura la variable `rebote`— no son independientes.
Repartirlos entre entrenamiento y prueba infla el resultado.

La partición honesta es temporal, que además es como se usaría el modelo de
verdad: entrenar con el pasado, predecir el futuro.

In [5]:
X = pd.get_dummies(
    d[['dist', 'angulo', 'dist_y', 'location_x', 'location_y',
       'asistido', 'rebote', 'situation', 'body_part']],
    columns=['situation', 'body_part'], dtype=float)
FEATS = list(X.columns)
y = d.gol.values

tr = d.temporada.isin(['2021-22', '2022-23', '2023-24']).values   # entrenamiento
va = (d.temporada == '2024-25').values                            # validación (early stopping)
te = (d.temporada == '2025-26').values                            # prueba, intocada

print(f'entrenamiento {tr.sum():6d} tiros (2021-22 a 2023-24)')
print(f'validación    {va.sum():6d} tiros (2024-25)')
print(f'prueba        {te.sum():6d} tiros (2025-26) · {y[te].sum()} goles')
print(f'\n{len(FEATS)} features: {FEATS}')

entrenamiento 135335 tiros (2021-22 a 2023-24)
validación     43108 tiros (2024-25)
prueba         43479 tiros (2025-26) · 4289 goles

15 features: ['dist', 'angulo', 'dist_y', 'location_x', 'location_y', 'asistido', 'rebote', 'situation_Direct Freekick', 'situation_From Corner', 'situation_Open Play', 'situation_Set Piece', 'body_part_Head', 'body_part_Left Foot', 'body_part_Other', 'body_part_Right Foot']


> **La columna `xg` no es una feature.** Es la predicción de Understat: usarla
> sería fuga de datos descarada. Es el rival a batir, y por eso no aparece en
> `FEATS`.

## 4. Tres modelos, de menos a más

1. **La tasa global** — predecir siempre 0.0996. Es el piso: cualquier cosa que
   no le gane, no sirve.
2. **Regresión logística con distancia y ángulo** — el modelo clásico de xG,
   interpretable y con dos variables.
3. **XGBoost con todo** — para ver cuánto suman el contexto y las interacciones
   sobre la geometría pura.

In [6]:
def evaluar(nombre, p):
    return {'Modelo': nombre,
            'Log-loss': log_loss(y[te], p), 'Brier': brier_score_loss(y[te], p),
            'ROC-AUC': roc_auc_score(y[te], p),
            'xG total': p.sum(), 'Goles reales': int(y[te].sum()),
            'Sesgo %': (p.sum() / y[te].sum() - 1) * 100}


filas = [evaluar('Tasa global (baseline)', np.full(te.sum(), y[tr].mean()))]

logit = LogisticRegression(max_iter=1000).fit(X.loc[tr, ['dist', 'angulo']], y[tr])
filas.append(evaluar('Logística (distancia + ángulo)',
                     logit.predict_proba(X.loc[te, ['dist', 'angulo']])[:, 1]))

xgb = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=5,
                    min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                    reg_lambda=1.0, eval_metric='logloss', early_stopping_rounds=50,
                    random_state=RNG, n_jobs=4)
xgb.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
p_xgb = xgb.predict_proba(X[te])[:, 1]
filas.append(evaluar(f'XGBoost ({xgb.best_iteration} árboles)', p_xgb))

p_und = d.xg.values[te]
filas.append(evaluar('xG de Understat (benchmark)', p_und))

print(f'logística — coeficientes: distancia {logit.coef_[0][0]:+.4f} · '
      f'ángulo {logit.coef_[0][1]:+.4f}\n')
pd.DataFrame(filas).round(4)

logística — coeficientes: distancia -0.0573 · ángulo +1.6199



,Modelo,Log-loss,Brier,ROC-AUC,xG total,Goles reales,Sesgo %
0,Tasa global (baseline),0.3221,0.0889,0.5000,4308.8658,4289,0.4632
1,Logística (distancia + ángulo),0.2915,0.0819,0.7186,4429.0242,4289,3.2647
2,XGBoost (1502 árboles),0.2597,0.0736,0.8044,4496.2236,4289,4.8315
3,xG de Understat (benchmark),0.2589,0.0733,0.7991,5005.9963,4289,16.7171


### Lectura de la tabla

**El modelo propio empata con el de producción.** Log-loss 0.2597 contra 0.2589
—una diferencia del 0.3%— y **le gana en discriminación**: AUC 0.804 contra
0.799. Con geometría y cuatro categóricas, sin ver un solo defensor ni la
posición del portero.

Dos cosas más que dice la tabla:

- **La geometría sola llega lejos, pero no hasta el final.** La logística con dos
  variables baja el log-loss de 0.322 a 0.292; XGBoost lo lleva a 0.260. Ese
  salto extra es el contexto: cabeza contra pie, córner contra jugada, rebote.
- **Y la columna `Sesgo %` tiene algo raro.** El xG de Understat suma 5.006
  contra 4.289 goles reales: **sobreestima un 16.7%**. El nuestro se pasa un
  4.8%. Eso no es un detalle de redondeo, y es el tema de la sección 6.

## 5. Calibración: la prueba que de verdad importa

Un xG no se usa para ordenar tiros, se usa para **sumarlos**. Cuando se dice "el
equipo generó 2.3 de xG", se está afirmando que esos tiros valían 2.3 goles. Por
eso lo que hay que exigirle no es que acierte cuál entra —eso es el AUC— sino que
**cuando dice 0.30, entre el 30%**.

Es la propiedad que casi nunca se grafica, y la que separa un xG usable de uno
decorativo.

In [7]:
cal = pd.DataFrame({'nuestro': p_xgb, 'understat': p_und, 'gol': y[te]})
cal['bin'] = pd.qcut(cal.nuestro, 12, labels=False, duplicates='drop')
curva = cal.groupby('bin').agg(n=('gol', 'size'), nuestro=('nuestro', 'mean'),
                               understat=('understat', 'mean'), real=('gol', 'mean'))

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 0.65], y=[0, 0.65], mode='lines', name='calibración perfecta',
                         line=dict(color=INK['axis'], width=1.5, dash='dash'), hoverinfo='skip'))
for col, nombre, color in [('nuestro', 'Nuestro XGBoost', SEQUENTIAL_BLUE[6]),
                           ('understat', 'xG de Understat', DIVERGING['neg'])]:
    fig.add_trace(go.Scatter(
        x=curva[col], y=curva['real'], mode='lines+markers', name=nombre,
        line=dict(color=color, width=2.5),
        marker=dict(size=9, line=dict(color=INK['surface'], width=1.5)),
        customdata=curva['n'],
        hovertemplate=f'<b>{nombre}</b><br>predice %{{x:.3f}}<br>entra %{{y:.3f}}'
                      '<br>(%{customdata} tiros)<extra></extra>'))
fig.update_layout(
    title=dict(text='Cuando el modelo dice 0.30, ¿entra el 30%?',
               subtitle=dict(text='Por encima de la diagonal el modelo se queda corto; '
                                  'por debajo, promete de más')),
    xaxis=dict(title='xG predicho (media del grupo)'),
    yaxis=dict(title='Goles reales (proporción)'),
    legend=dict(orientation='h', y=-0.16, x=0))
render_plot(fig, width=760, height=540)
curva.round(3)

,n,nuestro,understat,real
bin,,,,
0,3624,0.007,0.026,0.014
1,3623,0.014,0.035,0.019
2,3623,0.021,0.044,0.024
3,3623,0.028,0.049,0.026
4,3623,0.036,0.059,0.040
5,3624,0.047,0.067,0.043
6,3623,0.062,0.083,0.062
7,3623,0.082,0.100,0.075
8,3623,0.112,0.134,0.088


**Nuestra curva se pega a la diagonal; la de Understat vive por debajo de ella en
casi todo el recorrido** — es decir, promete más goles de los que entran. La
brecha es mayor en los tiros difíciles: donde nosotros predecimos 0.008 y entra
el 1.4%, Understat predice 0.027. Multiplicado por decenas de miles de tiros
malos, ahí está la mayor parte de esos 700 goles de más.

## 6. El hallazgo que no se buscaba

El sesgo de Understat no es cosa de una temporada.

In [8]:
todos = pd.read_csv('../data/processed/shots_all_seasons.csv', encoding='utf-8-sig')
todos = todos[(todos.result != 'Own Goal') & (todos.situation != 'Penalty')]
todos['gol'] = (todos.result == 'Goal').astype(int)

por_temp = todos.groupby('temporada').agg(tiros=('gol', 'size'), goles=('gol', 'sum'),
                                          xg_understat=('xg', 'sum'))
por_temp['sesgo_%'] = (por_temp.xg_understat / por_temp.goles - 1) * 100

por_liga = todos.groupby('liga').agg(goles=('gol', 'sum'), xg_understat=('xg', 'sum'))
por_liga['sesgo_%'] = (por_liga.xg_understat / por_liga.goles - 1) * 100

print(por_temp.round(1).to_string()); print()
print(por_liga.round(1).to_string())

           tiros  goles  xg_understat  sesgo_%
temporada                                     
2021-22    45022   4501        4798.2      6.6
2022-23    45111   4441        4881.4      9.9
2023-24    45202   4470        5024.9     12.4
2024-25    43108   4398        5039.8     14.6
2025-26    43479   4289        5006.0     16.7

                goles  xg_understat  sesgo_%
liga                                        
Bundesliga       4355        4652.5      6.8
La Liga          4324        4984.7     15.3
Ligue 1          4118        4636.8     12.6
Premier League   4956        5601.6     13.0
Serie A          4346        4874.8     12.2


In [9]:
fig = go.Figure()
fig.add_trace(go.Bar(x=por_temp.index, y=por_temp.goles, name='Goles reales',
                     marker=dict(color=SEQUENTIAL_BLUE[2], line=dict(width=0)),
                     hovertemplate='%{x}<br>%{y} goles<extra></extra>'))
fig.add_trace(go.Bar(x=por_temp.index, y=por_temp.xg_understat, name='xG de Understat',
                     marker=dict(color=DIVERGING['neg'], line=dict(width=0)),
                     hovertemplate='%{x}<br>%{y:.0f} de xG<extra></extra>'))
fig.add_trace(go.Scatter(x=por_temp.index, y=por_temp['sesgo_%'], name='Sobreestimación (%)',
                         yaxis='y2', mode='lines+markers',
                         line=dict(color=INK['secondary'], width=2.5),
                         marker=dict(size=10, line=dict(color=INK['surface'], width=2)),
                         hovertemplate='%{x}<br>sobreestima %{y:.1f}%<extra></extra>'))
fig.update_layout(
    barmode='group', bargap=0.3,
    title=dict(text='El xG de Understat se aleja de la realidad, año tras año',
               subtitle=dict(text='Mismos tiros, sin penales · la brecha pasa de 7% a 17% '
                                  'en cinco temporadas')),
    xaxis=dict(title=None),
    yaxis=dict(title='Goles / xG acumulado'),
    yaxis2=dict(title='Sobreestimación (%)', overlaying='y', side='right',
                showgrid=False, rangemode='tozero'),
    legend=dict(orientation='h', y=-0.16, x=0))
render_plot(fig, width=820, height=520)

**+7% en 2021-22, +17% en 2025-26, y creciendo de forma monótona.** Está en las
cinco ligas (entre +7% y +15%). Los goles reales bajan temporada a temporada
mientras su xG sube.

### Una hipótesis, y su refutación

La explicación natural: **el modelo envejece**. Si el fútbol cambia —mejores
porteros, distinta selección de tiro— un xG entrenado hace años sobreestima cada
vez más, y el sesgo crece con los años transcurridos.

Es comprobable con nuestro propio modelo: entrenarlo con **una sola temporada** y
probarlo siempre en 2025-26. Si la hipótesis es correcta, cuanto más vieja la
temporada de entrenamiento, mayor el sesgo.

In [10]:
filas = []
for s in ['2021-22', '2022-23', '2023-24', '2024-25']:
    m = (d.temporada == s).values
    mod = XGBClassifier(n_estimators=350, learning_rate=0.05, max_depth=5,
                        min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                        eval_metric='logloss', random_state=RNG, n_jobs=4).fit(X[m], y[m])
    p = mod.predict_proba(X[te])[:, 1]
    filas.append({'Entrenado solo con': s, 'Años de distancia': 2025 - int(s[:4]),
                  'xG predicho': p.sum(), 'Goles reales': int(y[te].sum()),
                  'Sesgo %': (p.sum() / y[te].sum() - 1) * 100,
                  'Log-loss': log_loss(y[te], p)})
pd.DataFrame(filas).round(3)

,Entrenado solo con,Años de distancia,xG predicho,Goles reales,Sesgo %,Log-loss
0,2021-22,4,4497.509766,4289,4.861,0.271
1,2022-23,3,4378.361816,4289,2.084,0.270
2,2023-24,2,4375.480957,4289,2.016,0.268
3,2024-25,1,4426.478027,4289,3.205,0.269


### La hipótesis explica 3 puntos de 17

Entrenar con la temporada más vieja (2021-22, cuatro años de distancia) da un
sesgo de **+4.9%**; entrenar con la más reciente da **+2.0-3.2%**. O sea que el
envejecimiento existe y va en la dirección esperada, pero aporta apenas **unos 3
puntos porcentuales**.

Understat se pasa **17**. Los otros 14 no son deriva del entorno: son del modelo.
No se puede saber de qué, porque no publican ni sus features ni su entrenamiento.

Conviene dejarlo dicho con precisión, sin exagerar el hallazgo:

- **No** significa que el xG de Understat "esté mal" para todo. Ordena tiros casi
  tan bien como el nuestro (AUC 0.799 vs 0.804): para comparar dos tiros, sirve.
- **Sí** significa que **no se puede sumar sin corregir**. Un "el equipo generó
  1.8 de xG" de Understat vale, en goles, más cerca de 1.55.
- Y sobre todo: **un número publicado por una fuente respetada no es la verdad.**
  Es otro modelo, con sus sesgos, y contrastarlo contra el resultado real —que es
  lo único que sí es verdad— cuesta cuatro líneas de código.

## 7. Qué aprendió el modelo

### El costo de cabecear, ahora sí

In [11]:
def predecir(dist_m, parte, situacion='Open Play', asistido=1, rebote=0):
    '''xG de un tiro de frente a la portería, a `dist_m` metros.'''
    fila = pd.DataFrame([{c: 0.0 for c in FEATS}])
    fila['dist'] = dist_m
    fila['dist_y'] = 0.0
    fila['angulo'] = np.arccos(np.clip((2 * dist_m ** 2 - PORTERIA ** 2) / (2 * dist_m ** 2), -1, 1))
    fila['location_x'], fila['location_y'] = 1 - dist_m / LARGO, 0.5
    fila['asistido'], fila['rebote'] = asistido, rebote
    fila[f'situation_{situacion}'] = 1.0
    fila[f'body_part_{parte}'] = 1.0
    return xgb.predict_proba(fila[FEATS])[0, 1]


distancias = np.arange(4, 31)
cabeza = [predecir(x, 'Head') for x in distancias]
pie = [predecir(x, 'Right Foot') for x in distancias]

fig = go.Figure()
for serie, nombre, color in [(pie, 'Pie derecho', SEQUENTIAL_BLUE[6]),
                             (cabeza, 'Cabeza', DIVERGING['neg'])]:
    fig.add_trace(go.Scatter(x=distancias, y=serie, mode='lines', name=nombre,
                             line=dict(color=color, width=3),
                             hovertemplate=f'<b>{nombre}</b><br>a %{{x}} m: '
                                           'xG %{y:.3f}<extra></extra>'))
fig.update_layout(
    title=dict(text='A igual distancia, un cabezazo vale mucho menos',
               subtitle=dict(text='Tiro de frente a la portería, jugada abierta · '
                                  'la tabla cruda daba 9.9% para los dos')),
    xaxis=dict(title='Distancia a la portería (m)'),
    yaxis=dict(title='xG predicho'),
    legend=dict(orientation='h', y=-0.16, x=0))
render_plot(fig, width=760, height=490)

for x in [6, 10, 16, 20]:
    h, p = predecir(x, 'Head'), predecir(x, 'Right Foot')
    print(f'  a {x:2d} m — cabeza {h:.3f} · pie {p:.3f} → el cabezazo vale el {h/p:.0%}')

  a  6 m — cabeza 0.607 · pie 0.671 → el cabezazo vale el 91%
  a 10 m — cabeza 0.143 · pie 0.333 → el cabezazo vale el 43%
  a 16 m — cabeza 0.026 · pie 0.164 → el cabezazo vale el 16%
  a 20 m — cabeza 0.020 · pie 0.086 → el cabezazo vale el 23%


El promedio crudo decía 9.9% contra 10.0%. El modelo, **a igual geometría**, dice
que a 6 metros un cabezazo vale el 91% de un remate con el pie, a 10 metros el
43%, y a 16 metros el **16%**. La penalización crece con la distancia hasta unos 16 m, que es justo lo que la
intuición futbolística dice y el `groupby` era incapaz de ver.

Más allá de ahí la curva deja de ser fiable y hasta se da vuelta (a 20 m el
cabezazo sale en 23%, por encima del 16% de 16 m). No es un hallazgo: es el
modelo extrapolando donde casi no hay datos — cabecear desde 20 metros es tan
raro que los árboles no tienen con qué cortar. La lectura honesta del gráfico
termina donde termina el área grande.

### El mapa de valor del tiro

In [12]:
nx, ny = 60, 44
xs = np.linspace(0.55, 0.995, nx)
ys = np.linspace(0.06, 0.94, ny)
malla = pd.DataFrame([(a, b) for b in ys for a in xs], columns=['location_x', 'location_y'])

mx, my = malla.location_x * LARGO, malla.location_y * ANCHO
malla['dist'] = np.hypot(LARGO - mx, ANCHO / 2 - my)
pa = np.hypot(LARGO - mx, ANCHO / 2 + PORTERIA / 2 - my)
pb = np.hypot(LARGO - mx, ANCHO / 2 - PORTERIA / 2 - my)
malla['angulo'] = np.arccos(np.clip((pa ** 2 + pb ** 2 - PORTERIA ** 2) / (2 * pa * pb), -1, 1))
malla['dist_y'] = (malla.location_y - 0.5).abs() * ANCHO
for c in FEATS:
    if c not in malla:
        malla[c] = 0.0
malla['asistido'] = 1.0
malla['situation_Open Play'] = 1.0
malla['body_part_Right Foot'] = 1.0

z = xgb.predict_proba(malla[FEATS])[:, 1].reshape(ny, nx)

fig = go.Figure(go.Heatmap(
    z=z, x=xs * LARGO, y=ys * ANCHO, colorscale=SEQUENTIAL_BLUE, zsmooth='best',
    colorbar=dict(title=dict(text='xG', side='right'), thickness=14, outlinewidth=0),
    hovertemplate='a %{x:.0f} m de la línea de fondo<br>%{y:.0f} m de ancho'
                  '<br>xG %{z:.3f}<extra></extra>'))
# área grande, área chica y portería
for x0, y0, x1, y1 in [(LARGO - 16.5, ANCHO / 2 - 20.16, LARGO, ANCHO / 2 + 20.16),
                       (LARGO - 5.5, ANCHO / 2 - 9.16, LARGO, ANCHO / 2 + 9.16)]:
    fig.add_shape(type='rect', x0=x0, y0=y0, x1=x1, y1=y1,
                  line=dict(color=INK['surface'], width=1.5), opacity=0.55)
fig.add_shape(type='line', x0=LARGO, y0=ANCHO / 2 - PORTERIA / 2, x1=LARGO,
              y1=ANCHO / 2 + PORTERIA / 2, line=dict(color=DIVERGING['neg'], width=4))
fig.update_layout(
    title=dict(text='El mapa de valor del tiro',
               subtitle=dict(text='xG de un remate con el pie derecho, de jugada y asistido · '
                                  'la línea roja es la portería')),
    xaxis=dict(title='Metros', constrain='domain'),
    yaxis=dict(title=None, scaleanchor='x', scaleratio=1, showticklabels=False))
render_plot(fig, width=800, height=560)

In [13]:
imp = (pd.Series(xgb.get_booster().get_score(importance_type='gain'))
       .sort_values(ascending=True))
fig = go.Figure(go.Bar(
    x=imp.values, y=imp.index, orientation='h',
    marker=dict(color=SEQUENTIAL_BLUE[5], line=dict(width=0)),
    hovertemplate='<b>%{y}</b><br>ganancia media %{x:.1f}<extra></extra>'))
fig.update_layout(
    title=dict(text='De qué se agarra el modelo',
               subtitle=dict(text='Ganancia media por corte · cabeza y rebote pesan más que '
                                  'la distancia cruda, que ya viene dentro del ángulo')),
    xaxis=dict(title='Ganancia'), yaxis=dict(title=None), margin=dict(l=190))
render_plot(fig, width=800, height=520)

El orden sorprende a primera vista: **`body_part_Head` y `rebote` pesan más que
`dist`**. No es que la distancia no importe — es que el **ángulo** ya la contiene
en buena medida (a más lejos, menos ángulo), así que una vez que el árbol cortó
por ángulo, la distancia aporta poco extra. Lo que sí aporta información nueva es
el contexto: si fue de cabeza, y si venía de un barullo en el área.

Que `rebote` esté segundo justifica haberlo construido: es la única variable de
secuencia disponible, es un proxy burdo —resolución de minuto, marca los dos
tiros por igual— y aun así es la segunda fuente de información del modelo.

## Conclusión

1. **Un xG competitivo se construye con geometría y cuatro categóricas.** El
   modelo propio empata en log-loss con el de Understat (0.2597 vs 0.2589) y le
   gana en AUC (0.804 vs 0.799), sin ver defensores ni portero. La mayor parte de
   la señal de un tiro está en dónde y cómo se remató.
2. **La calibración es lo que separa un xG usable de uno decorativo**, y es donde
   está la diferencia real entre los dos modelos. Ordenar tiros lo hacen los dos
   igual de bien; sumarlos, no.
3. **El xG de Understat sobreestima los goles un 17%, y el sesgo crece año a
   año.** Se propuso que fuera envejecimiento del modelo y **se midió que eso
   explica solo ~3 de los 17 puntos**. El resto queda sin explicación posible
   desde fuera, porque no publican su método.
4. **Un número de una fuente respetada no es la verdad, es otro modelo.**
   Contrastarlo contra el resultado real cuesta cuatro líneas, y en este caso
   cambia cómo hay que leer cualquier xG acumulado de Understat.

**Limitaciones**, en orden de importancia: no hay posición de portero ni de
defensores, que es la información que más falta y la que probablemente explica el
techo de ambos modelos; el proxy de rebote tiene resolución de minuto; y el
modelo no distingue el tipo de pase previo (centro, pase al hueco, atrás), que en
los modelos comerciales es una de las variables fuertes.

**Lo que habilita**: con un xG propio y calibrado se puede preguntar si la
**definición** es una habilidad — si un jugador supera sistemáticamente su xG año
tras año, o es ruido. Eso reusa la maquinaria de `ml_style_persistence.ipynb`
(ICC, correlación entre temporadas, test de nulo) sobre los 13.980
jugadores-temporada.